## Template Matching Approach
Here, I try to use a Template matching algorithm which accounts for some distortions. 

In [46]:
# %% Import necessary libraries
%load_ext autoreload
%autoreload 1
%aimport utils
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity
import os
import utils
import glob


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
def template_matching(A_file_path, B_file_path):
    A_name = os.path.splitext(os.path.basename(A_file_path))[0]
    B_name = os.path.splitext(os.path.basename(B_file_path))[0]

    # print(A_file_path)
    # print(B_file_path)
    images, keypoints, descriptors = utils.get_data(A_file_path, B_file_path)

    if len(keypoints[0]) <= 4 or len(keypoints[1]) <= 4:
        print("No keypoints found on one image.")
        return False

    kp_A = utils.plot_keypoints(images[0], keypoints[0], "Image A with Sift Features")
    kp_B = utils.plot_keypoints(images[1], keypoints[1], "Image B with Sift Features")

    index1 = 1
    index2 = 0

    matches = utils.calculate_matches(descriptors[index1], descriptors[index2])
    plt_matches = utils.plot_matches(images[index1],images[index2],keypoints[index1],keypoints[index2],matches,'Matches')

    trans, inliers = utils.get_transform(keypoints[index1],keypoints[index2],matches)
    if trans is None or inliers is None:
        return False
    inlier_matches = [matches[i] for i in inliers.T[0]]
    # print("Numbers of inliers found: ", len(inlier_matches))
    plt_inliers = utils.plot_matches(images[index1],images[index2],keypoints[index1],keypoints[index2],inlier_matches,'Inliers')

    if len(inlier_matches) < 40:
        # print("Not enough inliers found for matching images")
        return False

    border_size = 3
    border_color_bgr = (0, 0, 0)
    outputImage = cv2.copyMakeBorder(
        images[index1],
        border_size,
        border_size,
        border_size,
        border_size,
        cv2.BORDER_CONSTANT,
        value=border_color_bgr
    )
    im_out = cv2.warpPerspective(outputImage, trans,(images[index2].shape[1],images[index2].shape[0]))
    # plt_overlay = utils.plot_transformed_image(im_out,images[index2],'Overlay')

    full_path_A = "Exports/asdf/"+A_name+".png"
    full_path_B = "Exports/asdf/"+B_name+".png"

    im_out_A = cv2.cvtColor(im_out, cv2.COLOR_BGR2RGB)
    im_out_B = cv2.cvtColor(images[index2], cv2.COLOR_BGR2RGB)
    # cv2.imwrite(full_path_A, im_out_A)
    # cv2.imwrite(full_path_B, im_out_B)


    outFile = "Exports/comparisons/"+A_name+"AND"+B_name+".pdf"

    # utils.create_pdf_report_from_images(kp_A, kp_B, plt_matches, plt_inliers, plt_overlay, outFile)

    similarity = utils.orb_sim(im_out_A, im_out_B)
    print(similarity)

    return similarity >= 0.2


In [43]:
plt.rcParams['figure.figsize'] = [8, 5]
plt.rcParams['figure.dpi'] = 150

folder_A = "images/Top100/cleanBW"
folder_B = "images/Top100/frame"

image_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')

path_pattern_A = os.path.join(folder_A, '*')
files_A = [f for f in glob.glob(path_pattern_A)
           if os.path.isfile(f) and f.lower().endswith(image_extensions)]

path_pattern_B = os.path.join(folder_B, '*')
files_B = [f for f in glob.glob(path_pattern_B)
           if os.path.isfile(f) and f.lower().endswith(image_extensions)]

files_A.sort()
files_B.sort()

files_A_skewd = files_A.copy()
first_element = files_A_skewd.pop(0)
files_A_skewd.append(first_element)

# print(files_A)
# print(files_A_skewd)

files_A = files_A + files_A_skewd
files_B = files_B + files_B

res = []
tru = [True] * 100 + [False] * 100


print(files_A)
print(files_B)

for i in range(len(files_A)):
    res.append(template_matching(files_A[i], files_B[i]))

print(res)

"""
for A_file_path in files_A[1:15]:
    for B_file_path in files_B[1:15]:
        template_matching(A_file_path, B_file_path)
"""




['images/Top100/cleanBW/aivazovsky001.png', 'images/Top100/cleanBW/bonnard060.png', 'images/Top100/cleanBW/bosch-006.png', 'images/Top100/cleanBW/botticelli001.png', 'images/Top100/cleanBW/botticelli002.png', 'images/Top100/cleanBW/boucher044.png', 'images/Top100/cleanBW/bouguereau055.png', 'images/Top100/cleanBW/bruegel001.png', 'images/Top100/cleanBW/bruegel003.png', 'images/Top100/cleanBW/caillebotte001.png', 'images/Top100/cleanBW/caillebotte016.png', 'images/Top100/cleanBW/caravaggio011.png', 'images/Top100/cleanBW/caravaggio031.png', 'images/Top100/cleanBW/cezanne030.png', 'images/Top100/cleanBW/collier004.png', 'images/Top100/cleanBW/davidjl001.png', 'images/Top100/cleanBW/davidjl002.png', 'images/Top100/cleanBW/davidjl006.png', 'images/Top100/cleanBW/degas001.png', 'images/Top100/cleanBW/degas002.png', 'images/Top100/cleanBW/degas012.png', 'images/Top100/cleanBW/degas051.png', 'images/Top100/cleanBW/delacroix-005.png', 'images/Top100/cleanBW/duchamp-001.png', 'images/Top100/cle

'\nfor A_file_path in files_A[1:15]:\n    for B_file_path in files_B[1:15]:\n        template_matching(A_file_path, B_file_path)\n'

In [44]:
# 4. Initialize counters
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0

# 5. Compare lists element by element
for i in range(len(tru)):
    actual = tru[i]
    predicted = res[i]

    if actual is True and predicted is True:
        true_positives += 1
    elif actual is False and predicted is False:
        true_negatives += 1
    elif actual is False and predicted is True:
        # Actual is Negative, but predicted Positive -> False Positive
        false_positives += 1
    elif actual is True and predicted is False:
        # Actual is Positive, but predicted Negative -> False Negative
        false_negatives += 1

# 6. Print the results (Confusion Matrix Components)
print("\n--- Confusion Matrix Components ---")
print(f"True Positives (TP):  {true_positives}")
print(f"True Negatives (TN):  {true_negatives}")
print(f"False Positives (FP): {false_positives} (Type I Error)")
print(f"False Negatives (FN): {false_negatives} (Type II Error)")

# 7. Verification (Optional but recommended)
total_calculated = true_positives + true_negatives + false_positives + false_negatives
print(f"\nTotal items calculated: {total_calculated}")
print(f"Original list length:   {len(tru)}")

# 8. Calculate derived metrics (Optional)
print("\n--- Derived Metrics ---")
total_population = total_calculated

# Accuracy: (TP + TN) / Total
accuracy = (true_positives + true_negatives) / total_population if total_population > 0 else 0
print(f"Accuracy: {accuracy:.4f}")

# Precision: TP / (TP + FP) - How many selected positives are actually positive?
precision_denominator = true_positives + false_positives
precision = true_positives / precision_denominator if precision_denominator > 0 else 0
print(f"Precision: {precision:.4f}")

# Recall (Sensitivity): TP / (TP + FN) - How many actual positives were found?
recall_denominator = true_positives + false_negatives
recall = true_positives / recall_denominator if recall_denominator > 0 else 0
print(f"Recall (Sensitivity): {recall:.4f}")

# Specificity: TN / (TN + FP) - How many actual negatives were correctly identified?
specificity_denominator = true_negatives + false_positives
specificity = true_negatives / specificity_denominator if specificity_denominator > 0 else 0
print(f"Specificity: {specificity:.4f}")

# F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
f1_denominator = precision + recall
f1_score = 2 * (precision * recall) / f1_denominator if f1_denominator > 0 else 0
print(f"F1 Score: {f1_score:.4f}")



--- Confusion Matrix Components ---
True Positives (TP):  65
True Negatives (TN):  96
False Positives (FP): 4 (Type I Error)
False Negatives (FN): 35 (Type II Error)

Total items calculated: 200
Original list length:   200

--- Derived Metrics ---
Accuracy: 0.8050
Precision: 0.9420
Recall (Sensitivity): 0.6500
Specificity: 0.9600
F1 Score: 0.7692
